#  ### CUSTOMERS

In [0]:
from typing import List
from pyspark.sql.window import Window
from pyspark.sql import DataFrame
from pyspark.sql.types import *
from pyspark.sql.functions import *



In [0]:
from delta.tables import DeltaTable

In [0]:
class transformation:

    def dedup(self,df:DataFrame,dedup_cols:List,cdc:str):
        df=df.withColumn("dedupkey",concat(*dedup_cols))
        df=df.withColumn("dedupCounts",row_number().over(Window.partitionBy("dedupkey").orderBy(cdc)))
        df=df.filter(col('dedupCounts')==1)
        df=df.drop("dedupkey","dedupCounts")

        return df

    def processtimestamp(self,df):
        df=df.withColumn("process_timestamp",current_timestamp())
        return df

    def upsert(self,df,key_cols,table,cdc):
        merge_condition=" AND ".join([f"src.{i}=trg.{i}" for i in key_cols])
        dlt_obj=DeltaTable.forName(spark,f"pysparkdbt.silver.{table}")
        dlt_obj.alias("trg").merge(df.alias("src"),merge_condition)\
            .whenMatchedUpdateAll(condition=f"src.{cdc}>=trg.{cdc}")\
            .whenNotMatchedInsertAll()\
                .execute()
        return 1

In [0]:
import os
import sys

In [0]:
df_cust=spark.read.table("pysparkdbt.bronze.customers")

In [0]:
display(df_cust)

In [0]:
df_cust=df_cust.withColumn("domain",split(col("email"),"@")[1])

In [0]:
display(df_cust)

In [0]:
df_cust.withColumn("phone_number",regexp_replace("phone_number",r"[^0-9]",""))

In [0]:
df_cust=df_cust.withColumn("full_name",concat_ws(" ",col("first_name"),col("last_name")))
df_cust.drop("first_name","last_name")

In [0]:
cust_obj=transformation()

cust_trans=cust_obj.dedup(df_cust,['customer_id'],'last_updated_timestamp')
display(cust_trans)

In [0]:
df_cust=cust_obj.processtimestamp(cust_trans)
display(df_cust)

In [0]:

if not spark.catalog.tableExists("pysparkdbt.silver.customers"):
    df_cust.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.customers")
else:
    cust_obj.upsert(df_cust,["customer_id"],'customers','last_updated_timestamp')

In [0]:
%sql
SELECT count(*) FROM pysparkdbt.silver.customers

# ****# Drivers

In [0]:
df_drivers=spark.read.table("pysparkdbt.bronze.drivers")
display(df_drivers)

In [0]:

df_drivers=df_drivers.withColumn("phone_number",regexp_replace("phone_number",r"[^0-9]",""))

df_drivers=df_drivers.withColumn("full_name",concat_ws(" ",col('first_name'),col('last_name')))

df_drivers=df_drivers.drop("fist_name","last_name")

In [0]:
driver_obj=transformation()

df_drivers=driver_obj.dedup(df_drivers,['driver_id'],'last_updated_timestamp')
df_drivers=driver_obj.processtimestamp(df_drivers)

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.drivers"):

    df_drivers.write.format("delta")\
        .mode("append")\
            .saveAsTable("pysparkdbt.silver.drivers")
else:
    driver_obj.upsert(df_drivers,['driver_id'],'drivers','last_updated_timestamp')

In [0]:
%sql
SELECT count(*) FROM pysparkdbt.silver.drivers

# LOCATIONS

In [0]:
df_loc=spark.read.table("pysparkdbt.bronze.location")
display(df_loc)

In [0]:
loc_obj=transformation()
df_loc=loc_obj.dedup(df_loc,['location_id'],"last_updated_timestamp")
df_loc=loc_obj.processtimestamp(df_loc)

In [0]:
display(df_loc)

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.location"):

    df_loc.write.format("delta")\
        .mode("append")\
            .saveAsTable("pysparkdbt.silver.location")
else:
    loc_obj.upsert(df_loc,['location_id'],"location","last_updated_timestamp")

# Payment

In [0]:
df_pay=spark.read.table("pysparkdbt.bronze.payments")
display(df_pay)

In [0]:
df_py=df_pay.withColumn("online_payment_status",
                        when( ((col("payment_method")=='Card') & (col('payment_status')=='Success')),"online-success")
                        .when( ((col("payment_method")=='Card') & (col('payment_status')=='Failed')),"online-failure")
                        .when( ((col("payment_method")=='Card') & (col('payment_status')=='Pending')),"online-pending")
                        .otherwise("offline")
                       )

In [0]:
payment_obj=transformation()
df_py=payment_obj.dedup(df_py,['payment_id'],"last_updated_timestamp")
df_py=payment_obj.processtimestamp(df_py)



In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.payment"):

    df_py.write.format("delta")\
        .mode("append")\
            .saveAsTable("pysparkdbt.silver.payment")
else:
    payment_obj.upsert(df_py,['payment_id'],"payment","last_updated_timestamp")

In [0]:
%sql

select count(*) from pysparkdbt.silver.payment

Vehicles

In [0]:
df_veh=spark.read.table("pysparkdbt.bronze.vehicles")
display(df_veh)

In [0]:
df_veh=df_veh.withColumn("make",upper(col("make")))

In [0]:
veh_obj=transformation()
df_veh=veh_obj.dedup(df_veh,['vehicle_id'],'last_updated_timestamp')
df_veh=veh_obj.processtimestamp(df_veh)


In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.vehicles"):

    df_veh.write.format("delta")\
        .mode("append")\
            .saveAsTable("pysparkdbt.silver.vehicles")
else:
    veh_obj.upsert(df_veh,['vehicle_id'],"vehicles","last_updated_timestamp")

In [0]:
%sql
select count(*) from pysparkdbt.silver.vehicles